# Paper LLM scorer

Structured LLM scorer imported from `urbanomy.methods.agent`.


In [ ]:
from urbanomy.methods.agent import SingleAgentBaseline, init_llm
from urbanomy.methods.land_value_modeling import Evaluation

llm = init_llm("deepseek/deepseek-v4-flash")
baseline = SingleAgentBaseline(llm=llm, output_schema=Evaluation)

In [ ]:
import geopandas as gpd
basline_blocks = gpd.read_file('./data/blocks_agg_with_indicators.geojson')
basline_blocks.head()

In [ ]:
basline_blocks.loc[142]

In [ ]:
from catboost import CatBoostRegressor
model = CatBoostRegressor()
model.load_model('./data/catboost_land_value_no_services.cbm')  # модель на лог-цене
print(len(model.feature_names_))

In [ ]:
feature_cols = [
'residential','business','recreation','industrial','transport','special',
'agriculture','land_use','share','footprint_area','build_floor_area',
'living_area','non_living_area','population','site_area','fsi','gsi',
'mxi','l','morphotype','area_accessibility'
]
cat_features = ['land_use', 'morphotype']
numeric_feats = [c for c in feature_cols if c not in cat_features]
basline_blocks["id"] = basline_blocks.index

In [ ]:
basline_blocks['residential'] = basline_blocks['residential'].astype('float64')

In [ ]:
# удаляем старую колонку
basline_blocks = basline_blocks.drop(columns=["site_area"], errors="ignore").copy()

# пересчитываем площадь в метрической CRS
basline_blocks_metric = basline_blocks.to_crs(basline_blocks.estimate_utm_crs()).copy()
basline_blocks["site_area"] = basline_blocks_metric.geometry.area.values


In [ ]:
from urbanomy.methods.land_value_modeling import LandPriceEstimator

estimator = LandPriceEstimator(
    model=model,
    orig_features=numeric_feats+cat_features,
    categorical_features=cat_features,
    blocks=basline_blocks, #or blocks_198
)
blocks_pred = estimator.predict()
blocks_pred.head()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
# 1) Цена за сотку (100 м²)
blocks_pred["land_value_per_100m2"] = blocks_pred["land_value"] / blocks_pred["site_area"] * 100

# 2) Заменяем inf на NaN
blocks_pred = blocks_pred.replace([np.inf, -np.inf], np.nan)
blocks_pred = blocks_pred.fillna(0)

# 3) Удаляем выбросы: оставляем данные до 99-го перцентиля
p99 = blocks_pred["land_value_per_100m2"].quantile(0.95)
blocks_clean = blocks_pred[blocks_pred["land_value_per_100m2"] <= p99].copy()

print(blocks_clean["land_value_per_100m2"].describe())

# 4) Гистограмма после очистки
plt.figure()
blocks_clean["land_value_per_100m2"].dropna().hist(bins=50)
plt.xlabel("Цена за сотку (руб.)")
plt.ylabel("Частота")
plt.title("� аспределение цены за сотку")
plt.show()



In [ ]:
import matplotlib.pyplot as plt

blocks_clean.plot(
    column='land_value_per_100m2',
    legend=True,
    figsize=(20,20),
    cmap='coolwarm',
    edgecolor='black',   # <-- цвет границы
    linewidth=0.2        # <-- толщина границы
).set_axis_off()
plt.title('Карта стоимости земельных участков за сотку (руб.)', fontsize=16)

plt.show()

In [ ]:
import matplotlib.pyplot as plt

blocks_clean.plot(
    column='land_value',
    legend=True,
    figsize=(20,20),
    cmap='coolwarm',
    edgecolor='black',   # <-- цвет границы
    linewidth=0.2        # <-- толщина границы
).set_axis_off()
plt.title('Карта стоимости земельных участков (руб.)', fontsize=16)

plt.show()

# Выбор сценария развития

In [ ]:
blocks_clean["id"] = blocks_clean.index

In [ ]:
target_id = 86

In [ ]:
import matplotlib.pyplot as plt

target_id = 86
target_block = blocks_clean.loc[blocks_clean["id"] == target_id]

fig, ax = plt.subplots(figsize=(25, 35))
blocks_clean.plot(ax=ax, color="lightgrey", edgecolor="white", linewidth=0.6)
target_block.plot(ax=ax, color="none", edgecolor="gold", linewidth=5.5)
target_block.centroid.plot(ax=ax, color="red", markersize=30, zorder=3)

# ax.set_title(f"Изменяемый квартал (id={target_id})")
ax.axis("off")
plt.show()


## NSGA II (без LLM)

In [ ]:
from pathlib import Path

import pandas as pd
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.optimize import minimize

from urbanomy.methods.land_value_modeling import DistrictProblem, build_pareto_front_dataframe

notebook_dir = Path("examples") if Path("examples/paper.ipynb").exists() else Path(".")
output_dir = notebook_dir / "nsga_2_without_llm"
legacy_output_dirs = [output_dir, notebook_dir / "paper_data"]
output_dir.mkdir(exist_ok=True)

site_area = float(blocks_clean.loc[blocks_clean["id"] == target_id, "site_area"].iloc[0])

constraints = {
    "footprint_area": {"type": "float", "min": 0.0, "max": 0.1 * site_area},
    "l": {"type": "float", "min": 1.0, "max": 10.0},
    "mxi": {"type": "float", "min": 0.1, "max": 1.0},

    "residential": {"type": "float", "min": 0.0, "max": 1.0},
    "business": {"type": "float", "min": 0.0, "max": 1.0},
    "recreation": {"type": "float", "min": 0.0, "max": 1.0},
    "industrial": {"type": "float", "min": 0.0, "max": 1.0},
    "transport": {"type": "float", "min": 0.0, "max": 1.0},
    "special": {"type": "float", "min": 0.0, "max": 1.0},
    "agriculture": {"type": "float", "min": 0.0, "max": 1.0},
}


def seed_done(seed: int) -> bool:
    return any(
        (folder / f"no_llm_optimization_log_{seed}.jsonl").exists()
        and (folder / f"no_llm_pareto_front_{seed}.jsonl").exists()
        for folder in legacy_output_dirs
    )


for seed in [seed for seed in range(41, 42)]:
    problem_no_llm = DistrictProblem(
        blocks=blocks_clean,
        model=model,
        estimator_kwargs={
            "orig_features": numeric_feats + cat_features,
            "categorical_features": cat_features,
        },
        constraints=constraints,
        target_id=target_id,
        strategic_alignment_scorer=None,
        log_optimization=True,
    )

    algorithm = NSGA2(
        pop_size=20,
        eliminate_duplicates=True,
    )

    res_no_llm = minimize(
        problem_no_llm,
        algorithm,
        ('n_gen', 30),
        seed=seed,
        verbose=True,
        save_history=True,
    )

    no_llm_log_df = pd.DataFrame(problem_no_llm.optimization_log)
    no_llm_pareto_df = build_pareto_front_dataframe(
        res=res_no_llm,
        problem=problem_no_llm,
        scenario_prefix=f"no_llm_pareto_{seed}",
    )

    no_llm_log_df.to_json(output_dir / f"no_llm_optimization_log_{seed}.jsonl", orient="records", lines=True, force_ascii=False)
    no_llm_pareto_df.to_json(output_dir / f"no_llm_pareto_front_{seed}.jsonl", orient="records", lines=True, force_ascii=False)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

optimizer = problem_no_llm

# 1) Базовая стоимость "до"
baseline_land_value = optimizer.evaluate_catboost(
    geonome=optimizer.blocks,
    model=optimizer.model,
    orig_features=optimizer.estimator_kwargs["orig_features"],
    cat_features=optimizer.estimator_kwargs["categorical_features"],
    radius_list=None,
)

# 2) Собираем ВСЕ решения: из history (если optimize(..., save_history=True)), иначе из res.X/res.F
X_all, F_all = [], []
if getattr(res_no_llm, "history", None):
    for alg in res_no_llm.history:
        pop = alg.pop
        Xi, Fi = pop.get("X"), pop.get("F")
        if Xi is not None and Fi is not None and len(Xi):
            X_all.append(np.asarray(Xi))
            F_all.append(np.asarray(Fi))

if X_all:
    X = np.vstack(X_all)
    F = np.vstack(F_all)
else:
    X = np.asarray(res_no_llm.X)
    F = np.asarray(res_no_llm.F)

# 3) Цели
land_value_total = -F[:, 0]
admin_gain = land_value_total - baseline_land_value
investor_npv = -F[:, 1]

# 4) LANDUSE для каждого решения (через тот же repair, что в оптимизаторе)
landuse_labels = []
for genome_vec in X:
    changes = {name: genome_vec[j] for j, name in enumerate(optimizer.var_names)}
    repaired = optimizer._repair_genome(changes)   # использует логику из вашей задачи
    lu = repaired["land_use"]
    lu_name = getattr(lu, "name", str(lu).split(".")[-1])
    landuse_labels.append(lu_name)

landuse_labels = np.array(landuse_labels)
unique_lu = np.unique(landuse_labels)

# 5) Цвета + легенда
cmap = plt.get_cmap("tab10", len(unique_lu))
color_map = {lu: cmap(i) for i, lu in enumerate(unique_lu)}

plt.figure(figsize=(10, 6))

for lu in unique_lu:
    m = landuse_labels == lu
    plt.scatter(
        investor_npv[m],
        admin_gain[m],
        s=35,
        alpha=0.8,
        color=color_map[lu],
        label=f"LANDUSE: {lu}",
    )

# (опционально) поверх — линия самого Pareto-фронта из res_no_llm.F
pf_x = -res_no_llm.F[:, 1]
pf_y = -res_no_llm.F[:, 0] - baseline_land_value
order = np.argsort(pf_x)
plt.plot(pf_x[order], pf_y[order], color="black", lw=1.5, alpha=0.7, label="Pareto front")

plt.xlabel("NPV инвестора (X), руб.")
plt.ylabel("Прирост общей стоимости земли (Y), руб.")
plt.title("Парето фронт")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()   

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

notebook_dir = Path("examples") if Path("examples/NSGA2.ipynb").exists() else Path(".")
no_llm_dir = notebook_dir / "nsga_2_without_llm"
llm_dir = notebook_dir / "nsga_2_llm_prompts"

SEEDS = range(32, 42)  # 10 сидов: 32..41. Если надо 33..42, замени на range(33, 43)

seed_fronts = {
    seed: pd.read_json(no_llm_dir / f"no_llm_pareto_front_{seed}.jsonl", lines=True)
    for seed in SEEDS
}

prompt_fronts = {
    prompt_id: pd.read_json(llm_dir / f"prompt_{prompt_id}" / f"prompt_{prompt_id}_pareto_front.jsonl", lines=True)
    for prompt_id in range(1, 6)
}

plt.figure(figsize=(12, 8))

for seed, df in seed_fronts.items():
    df = df.sort_values("investor_npv")
    plt.scatter(
        df["investor_npv"],
        df["land_value_gain"],
        s=18,
        alpha=0.45,
        color="gray",
        label=f"no LLM seed {seed}",
    )
prompt_cmap = plt.get_cmap("Dark2", len(prompt_fronts))
for i, (prompt_id, df) in enumerate(prompt_fronts.items()):
    df = df.sort_values("investor_npv")
    plt.scatter(
        df["investor_npv"],
        df["land_value_gain"],
        s=28,
        alpha=0.9,
        marker="D",
        color=prompt_cmap(i),
        label=f"LLM prompt {prompt_id}",
    )

plt.xlabel("NPV инвестора (X), руб.")
plt.ylabel("Прирост общей стоимости земли (Y), руб.")
plt.title("Pareto fronts: 10 no-LLM seeds vs NSGA-II + LLM по 5 промптам")
plt.grid(True, alpha=0.3)
plt.legend(title="Фронт", ncol=3, fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd

from urbanomy.methods.land_value_modeling import (
    ScenarioTEPModifier,
    LandPriceEstimator,
    plot_scenario_impact,
)


optimizer = problem_no_llm
result = res_no_llm
F_front = np.atleast_2d(np.asarray(result.F))
X_front = np.atleast_2d(np.asarray(result.X))

baseline_land_value = optimizer.baseline_land_value()

idx = int(np.argmin(F_front[:, 1]))   # максимум investor_npv

params_optimal = {p: X_front[idx][i] for i, p in enumerate(optimizer.constraints.keys())}
params_repaired = optimizer._repair_genome(params_optimal)

metrics = {
    "params_repaired": params_repaired,
    "land_value_after": float(-F_front[idx, 0]),
    "land_value_gain": float(-F_front[idx, 0] - baseline_land_value),
    "investor_npv": float(-F_front[idx, 1]),
}
metrics

# Базовые кварталы должны быть теми же, что использовал оптимизатор
blocks_before = optimizer.blocks.copy()

# Помечаем проектный квартал
blocks_before["is_project"] = False
blocks_before.loc[blocks_before["id"] == target_id, "is_project"] = True

# 1) Применяем оптимизационный сценарий к базовым кварталам
modifier = ScenarioTEPModifier(blocks_before)
blocks_after = modifier.apply(target_id, params_repaired)
blocks_after["is_project"] = False
blocks_after.loc[blocks_after["id"] == target_id, "is_project"] = True

# 2) Считаем стоимость "до" и "после" тем же способом, что и в DistrictProblem
estimator_kwargs = dict(
    model=optimizer.model,
    orig_features=optimizer.estimator_kwargs["orig_features"],
    categorical_features=optimizer.estimator_kwargs["categorical_features"],
    use_service_features=False,
)

baseline_estimator = LandPriceEstimator(
    blocks=blocks_before,
    **estimator_kwargs,
)
blocks_before_pred = baseline_estimator.predict()

scenario_estimator = LandPriceEstimator(
    blocks=blocks_after,
    **estimator_kwargs,
)
blocks_after_pred = scenario_estimator.predict()

for df in (blocks_before_pred, blocks_after_pred):
    df["land_value_per_100m2"] = np.where(
        pd.to_numeric(df["site_area"], errors="coerce") > 0,
        pd.to_numeric(df["land_value"], errors="coerce")
        / pd.to_numeric(df["site_area"], errors="coerce") * 100,
        np.nan,
    )

baseline_cols = blocks_before_pred[
    ["id", "land_value", "land_value_per_100m2"]
].rename(
    columns={
        "land_value": "land_value_before",
        "land_value_per_100m2": "land_value_before_per_100m2",
    }
)

blocks_full_value = blocks_after_pred.merge(
    baseline_cols,
    on="id",
    how="left",
    validate="one_to_one",
)

blocks_full_value["d_rub"] = (
    blocks_full_value["land_value"] - blocks_full_value["land_value_before"]
)

blocks_full_value["land_value_delta_pct"] = np.where(
    pd.to_numeric(blocks_full_value["land_value_before"], errors="coerce") > 0,
    (blocks_full_value["land_value"] / blocks_full_value["land_value_before"] - 1.0) * 100,
    np.nan,
)

blocks_full_value = blocks_full_value.replace([np.inf, -np.inf], np.nan)

sum_before = float(blocks_full_value["land_value_before"].sum())
sum_after = float(blocks_full_value["land_value"].sum())
sum_delta = sum_after - sum_before
sum_delta_pct = (sum_after / sum_before - 1.0) * 100 if sum_before > 0 else np.nan

print("Изменение стоимости земли по всем кварталам:")
print(f" • Сумма до:    {sum_before:,.0f} ₽".replace(",", " "))
print(f" • Сумма после: {sum_after:,.0f} ₽".replace(",", " "))
print(f" • Изм., ₽:     {sum_delta:+,.0f} ₽".replace(",", " "))
print(f" • Изм., %:     {sum_delta_pct:+.2f}%")

target_row = blocks_full_value.loc[blocks_full_value["id"] == target_id].iloc[0]
print("\nИзменяемый квартал:")
print(
    f" • До: {target_row['land_value_before']:,.0f} ₽ | "
    f"После: {target_row['land_value']:,.0f} ₽ | "
    f"Δ: {target_row['d_rub']:+,.0f} ₽ | "
    f"Δ%: {target_row['land_value_delta_pct']:+.2f}%".replace(",", " ")
)

scenario_result = plot_scenario_impact(
    blocks=blocks_full_value,
    target_idx=target_id,
    target_id_column="id",
    print_summary=False,
    print_quarter_stats=False,
    figsize=(25, 35),
)

